# Comprehensive Pandas Guide
## From Basics to Advanced Data Manipulation

**Kajola Gbenga**
**Data Scientist & AI Engineer**

---

This notebook is a complete, line-by-line commented reference to the pandas library - from
creating your first Series to advanced reshaping, merging, and performance techniques. It
uses the same retail sales dataset from the earlier EDA notebook so every concept is
demonstrated on realistic data, not toy examples.

**Topics covered:**

1. What is pandas and why use it
2. Core data structures: Series and DataFrame
3. Loading and inspecting data
4. Selecting and indexing data
5. Filtering data (boolean indexing)
6. Sorting data
7. Handling missing data
8. Modifying data: adding, dropping, renaming columns
9. Apply, map, and applymap
10. Grouping and aggregation (groupby)
11. Pivot tables and cross-tabulation
12. Merging, joining, and concatenating
13. Reshaping data: melt, pivot, stack/unstack
14. String operations
15. Date and time handling
16. Categorical data
17. Window functions (rolling, expanding, cumulative)
18. MultiIndex (hierarchical indexing)
19. Exporting data
20. Performance tips and best practices


## 0. Setup - Import Pandas and Load Sample Data

In [1]:
# The standard convention is to import pandas as "pd" - this alias is used everywhere
# in the pandas ecosystem and in virtually every tutorial, so we follow it here too
import pandas as pd
import numpy as np

# Display settings so output is easy to read in this notebook
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)

# Check the installed pandas version - useful to know since some methods/parameters
# differ between pandas versions
print("Pandas version:", pd.__version__)

# Load the same retail dataset used in the earlier EDA notebook, so every pandas
# concept below is demonstrated on realistic, familiar data
df = pd.read_csv("retail_business_dataset.csv", parse_dates=["OrderDate"])
print(f"Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Pandas version: 3.0.2
Loaded dataset: 6,045 rows x 18 columns


,OrderID,OrderDate,Region,Store,ProductCategory,UnitPrice,Quantity,DiscountPercent,GrossSales,NetSales,Cost,Profit,PaymentMethod,CustomerSegment,CustomerAge,CustomerGender,SatisfactionScore,DeliveryDays
0,ORD103076,2023-03-21,South,Store_06,Electronics,374.09,1,15.0,374.09,2934.99121,217.65,1257.31554,Cash,New,45.0,Female,3.5,1.0
1,ORD104130,2023-01-07,South,Store_19,Beauty,47.48,1,0.0,47.48,47.48000,32.54,14.94000,Cash,Returning,40.0,Female,3.7,3.0
2,ORD104684,2024-01-13,West,Store_03,Groceries,21.36,2,0.0,42.72,42.72000,23.64,19.08000,Bank Transfer,New,31.0,Female,2.1,1.0
3,ORD103293,2023-07-31,West,Store_06,Clothing,184.86,4,0.0,887.33,887.33000,400.22,487.11000,Debit Card,Returning,45.0,Female,5.0,0.0
4,ORD102397,2023-07-31,Central,Store_19,Home & Living,51.84,1,0.0,62.21,62.21000,36.57,25.64000,Credit Card,Returning,37.0,Male,2.6,4.0


## 1. What Is Pandas and Why Use It

**Pandas** ("Panel Data") is Python's core library for working with structured/tabular data.
It gives Python two fundamentally important data structures:

- **Series** - a single labeled, one-dimensional array (like one column of a spreadsheet)
- **DataFrame** - a labeled, two-dimensional table (like a full spreadsheet or SQL table)

Pandas is built on top of NumPy, which gives it fast, vectorized numerical operations, while
adding labels (row/column names), mixed data types per column, and a huge library of
built-in operations for reading, cleaning, transforming, aggregating, and writing data.

**Why not just use plain Python lists/dictionaries or NumPy arrays?**
- Lists/dicts have no built-in concept of aligning data by row/column labels
- NumPy arrays require a single data type for the whole array (pandas allows mixed types per column)
- Pandas has native, one-line support for reading CSV/Excel/SQL/JSON, handling missing data,
  grouping, merging, pivoting, and time series - all things that would take many lines of
  raw Python to implement correctly.

## 2. Core Data Structures: Series and DataFrame

In [2]:
# A Series is a one-dimensional labeled array. Here we build one from a plain Python list.
# pandas automatically assigns a default integer index (0, 1, 2, ...) unless we specify one.
prices = pd.Series([19.99, 45.50, 12.00, 99.99], name="UnitPrice")
print(prices)
print("\nType:", type(prices))

0    19.99
1    45.50
2    12.00
3    99.99
Name: UnitPrice, dtype: float64

Type: <class 'pandas.Series'>


In [3]:
# A Series with a custom, explicit index (like a dictionary with ordered keys)
store_revenue = pd.Series(
    data=[125000, 98000, 143000],
    index=["Store_01", "Store_02", "Store_03"],
    name="Revenue"
)
print(store_revenue)

# Access a single value by its label, exactly like a dictionary lookup
print("\nStore_02 revenue:", store_revenue["Store_02"])

Store_01    125000
Store_02     98000
Store_03    143000
Name: Revenue, dtype: int64

Store_02 revenue: 98000


In [4]:
# A DataFrame can be built directly from a dictionary of equal-length lists -
# each key becomes a column name, each list becomes that column's values
sample_df = pd.DataFrame({
    "Product": ["Laptop", "Phone", "Tablet"],
    "Price": [999.99, 699.99, 399.99],
    "InStock": [True, False, True]
})
print(sample_df)
print("\nType:", type(sample_df))

# Every column of a DataFrame is itself a Series
print("\nType of a single column:", type(sample_df["Price"]))

  Product   Price  InStock
0  Laptop  999.99     True
1   Phone  699.99    False
2  Tablet  399.99     True

Type: <class 'pandas.DataFrame'>

Type of a single column: <class 'pandas.Series'>


In [5]:
# Key structural attributes every DataFrame has
print("Shape (rows, columns):", df.shape)
print("Number of dimensions:", df.ndim)
print("Total number of elements:", df.size)
print("Index (row labels):", df.index)
print("Columns:", list(df.columns[:5]), "...")

Shape (rows, columns): (6045, 18)
Number of dimensions: 2
Total number of elements: 108810
Index (row labels): RangeIndex(start=0, stop=6045, step=1)
Columns: ['OrderID', 'OrderDate', 'Region', 'Store', 'ProductCategory'] ...


## 3. Loading and Inspecting Data

In [6]:
# pandas can read many file formats with a single function call:
#   pd.read_csv()      -> CSV files
#   pd.read_excel()     -> Excel files (.xlsx, .xls)
#   pd.read_json()      -> JSON files
#   pd.read_sql()       -> SQL database queries
#   pd.read_parquet()   -> Parquet files
# We already loaded our CSV above with parse_dates=["OrderDate"], which tells pandas to
# convert that column straight to a datetime type during loading, rather than as a separate step.

# .head(n) / .tail(n) - first/last n rows (default n=5)
df.head(3)

,OrderID,OrderDate,Region,Store,ProductCategory,UnitPrice,Quantity,DiscountPercent,GrossSales,NetSales,Cost,Profit,PaymentMethod,CustomerSegment,CustomerAge,CustomerGender,SatisfactionScore,DeliveryDays
0,ORD103076,2023-03-21,South,Store_06,Electronics,374.09,1,15.0,374.09,2934.99121,217.65,1257.31554,Cash,New,45.0,Female,3.5,1.0
1,ORD104130,2023-01-07,South,Store_19,Beauty,47.48,1,0.0,47.48,47.48000,32.54,14.94000,Cash,Returning,40.0,Female,3.7,3.0
2,ORD104684,2024-01-13,West,Store_03,Groceries,21.36,2,0.0,42.72,42.72000,23.64,19.08000,Bank Transfer,New,31.0,Female,2.1,1.0


In [7]:
# .info() - a compact summary: column names, non-null counts, dtypes, memory usage.
# This is usually the very first command to run on any newly loaded DataFrame.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6045 entries, 0 to 6044
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   OrderID            6045 non-null   str           
 1   OrderDate          6045 non-null   datetime64[us]
 2   Region             6045 non-null   str           
 3   Store              5985 non-null   str           
 4   ProductCategory    6045 non-null   str           
 5   UnitPrice          6045 non-null   float64       
 6   Quantity           6045 non-null   int64         
 7   DiscountPercent    5923 non-null   float64       
 8   GrossSales         6045 non-null   float64       
 9   NetSales           6045 non-null   float64       
 10  Cost               6045 non-null   float64       
 11  Profit             6045 non-null   float64       
 12  PaymentMethod      6045 non-null   str           
 13  CustomerSegment    6045 non-null   str           
 14  CustomerAge        

In [8]:
# .describe() - summary statistics for numeric columns (count, mean, std, quartiles)
df.describe()

,OrderDate,UnitPrice,Quantity,DiscountPercent,GrossSales,NetSales,Cost,Profit,CustomerAge,SatisfactionScore,DeliveryDays
count,6045,6045.000000,6045.000000,5923.000000,6045.000000,6045.000000,6045.000000,6045.000000,5804.000000,5684.000000,5863.000000
mean,2023-12-30 21:06:20.545905,209.570543,1.561456,4.651359,372.540744,366.759628,212.395754,146.658770,37.523777,3.946481,3.537779
min,2023-01-01 00:00:00,0.000000,-1.000000,0.000000,2.050000,1.750000,0.990000,0.600000,-5.000000,1.000000,0.000000
25%,2023-07-01 00:00:00,39.460000,1.000000,0.000000,54.540000,52.440000,31.190000,20.670000,28.000000,3.400000,2.000000
50%,2023-12-29 00:00:00,98.150000,1.000000,0.000000,145.350000,139.310000,83.570000,53.900000,37.000000,4.000000,3.000000
75%,2024-07-02 00:00:00,259.210000,2.000000,10.000000,418.730000,401.900000,234.760000,158.970000,46.000000,4.600000,5.000000
max,2024-12-30 00:00:00,1199.670000,5.000000,25.000000,6546.180000,14448.077792,4304.490000,6258.437562,150.000000,5.000000,11.000000
std,NaN,268.487167,0.925091,6.890922,597.549193,637.339294,346.679087,256.472798,13.023883,0.784809,1.952759


In [9]:
# .dtypes - just the data type of each column, without the rest of .info()'s output
df.dtypes

OrderID                         str
OrderDate            datetime64[us]
Region                          str
Store                           str
ProductCategory                 str
UnitPrice                   float64
Quantity                      int64
DiscountPercent             float64
GrossSales                  float64
NetSales                    float64
Cost                        float64
Profit                      float64
PaymentMethod                   str
CustomerSegment                 str
CustomerAge                 float64
CustomerGender                  str
SatisfactionScore           float64
DeliveryDays                float64
dtype: object

In [10]:
# .nunique() - number of unique values in each column - quick way to spot
# ID-like columns (very high uniqueness) vs categorical columns (very low uniqueness)
df.nunique()

OrderID              6000
OrderDate             728
Region                  9
Store                  20
ProductCategory        11
UnitPrice            5405
Quantity                6
DiscountPercent         6
GrossSales           5627
NetSales             5614
Cost                 5375
Profit               5112
PaymentMethod           5
CustomerSegment         3
CustomerAge            70
CustomerGender          3
SatisfactionScore      41
DeliveryDays           12
dtype: int64

## 4. Selecting and Indexing Data

Pandas offers several ways to select data, and it is important to know which one to use when:

- `df["col"]` or `df.col` - select a single column (returns a Series)
- `df[["col1", "col2"]]` - select multiple columns (returns a DataFrame)
- `df.loc[]` - select by **label** (row/column names)
- `df.iloc[]` - select by **integer position** (like plain Python list indexing)

In [11]:
# Select a single column - returns a Series
region_col = df["Region"]
print(type(region_col))
region_col.head()

<class 'pandas.Series'>


0      South
1      South
2       West
3       West
4    Central
Name: Region, dtype: str

In [12]:
# Select multiple columns at once - pass a LIST of column names, returns a DataFrame
subset = df[["OrderID", "Region", "NetSales"]]
subset.head()

,OrderID,Region,NetSales
0,ORD103076,South,2934.99121
1,ORD104130,South,47.48000
2,ORD104684,West,42.72000
3,ORD103293,West,887.33000
4,ORD102397,Central,62.21000


In [13]:
# .loc[] selects by LABEL - both rows and columns can be specified as [row_labels, col_labels]
# Here: rows 0 through 4 (inclusive on both ends when using .loc with labels),
# and only the "Region" and "NetSales" columns
df.loc[0:4, ["Region", "NetSales"]]

,Region,NetSales
0,South,2934.99121
1,South,47.48000
2,West,42.72000
3,West,887.33000
4,Central,62.21000


In [14]:
# .iloc[] selects by INTEGER POSITION - behaves like standard Python slicing
# (end index is EXCLUSIVE, unlike .loc with labels)
# Here: the first 5 rows (positions 0-4) and the first 3 columns (positions 0-2)
df.iloc[0:5, 0:3]

,OrderID,OrderDate,Region
0,ORD103076,2023-03-21,South
1,ORD104130,2023-01-07,South
2,ORD104684,2024-01-13,West
3,ORD103293,2023-07-31,West
4,ORD102397,2023-07-31,Central


In [15]:
# .loc[] can also select a single cell directly by row label and column name
single_value = df.loc[0, "NetSales"]
print("Single cell value (row 0, NetSales):", single_value)

# .at[] is a faster alternative to .loc[] specifically for single-cell access
single_value_fast = df.at[0, "NetSales"]
print("Same value via .at[]:", single_value_fast)

Single cell value (row 0, NetSales): 2934.991210432297
Same value via .at[]: 2934.991210432297


## 5. Filtering Data (Boolean Indexing)

Filtering in pandas works by building a boolean (True/False) mask, one value per row, and
then using that mask to keep only the rows where it is True.

In [16]:
# Simple single-condition filter: all orders from the "North" region
# df["Region"] == "North" creates a Series of True/False values (the "mask")
# Passing that mask inside df[...] keeps only the True rows
north_orders = df[df["Region"] == "North"]
print(f"North region orders: {len(north_orders):,} out of {len(df):,} total")
north_orders.head(3)

North region orders: 1,329 out of 6,045 total


,OrderID,OrderDate,Region,Store,ProductCategory,UnitPrice,Quantity,DiscountPercent,GrossSales,NetSales,Cost,Profit,PaymentMethod,CustomerSegment,CustomerAge,CustomerGender,SatisfactionScore,DeliveryDays
6,ORD101940,2023-05-01,North,Store_14,Groceries,52.73,5,0.0,263.65,263.65,120.80,142.85,Mobile Wallet,VIP,45.0,Male,4.3,5.0
13,ORD103329,2024-10-31,North,Store_05,Electronics,627.86,3,0.0,1883.58,1883.58,1260.57,623.01,Mobile Wallet,New,32.0,Female,5.0,4.0
15,ORD101178,2023-07-09,North,Store_14,Clothing,188.74,2,0.0,452.98,452.98,323.85,129.13,Mobile Wallet,Returning,18.0,Male,5.0,3.0


In [17]:
# Multiple conditions combined with & (AND) / | (OR).
# IMPORTANT: each individual condition MUST be wrapped in parentheses when combining them -
# this is a very common beginner mistake, since Python's operator precedence would otherwise
# try to evaluate & before == and raise an error.
high_value_electronics = df[(df["ProductCategory"] == "Electronics") & (df["NetSales"] > 500)]
print(f"High-value Electronics orders: {len(high_value_electronics):,}")

# OR condition: orders that are EITHER from the West region OR paid by Cash
west_or_cash = df[(df["Region"] == "West") | (df["PaymentMethod"] == "Cash")]
print(f"West region OR Cash payment orders: {len(west_or_cash):,}")

High-value Electronics orders: 850
West region OR Cash payment orders: 2,189


In [18]:
# .isin() - filter rows where a column's value is any one of several options
# (avoids writing a long chain of == comparisons joined by |)
selected_categories = df[df["ProductCategory"].isin(["Electronics", "Sports"])]
print(f"Electronics or Sports orders: {len(selected_categories):,}")

# ~ negates a boolean mask (NOT). Here: every category EXCEPT Groceries
not_groceries = df[~(df["ProductCategory"] == "Groceries")]
print(f"Non-Grocery orders: {len(not_groceries):,}")

Electronics or Sports orders: 1,734
Non-Grocery orders: 4,308


In [19]:
# .query() - an alternative, often more readable way to filter, using a string expression
# instead of the bracket-and-mask syntax above. Especially handy for longer conditions.
result = df.query("ProductCategory == 'Clothing' and NetSales > 100 and Quantity >= 2")
print(f"Matching orders: {len(result):,}")
result.head(3)

Matching orders: 313


,OrderID,OrderDate,Region,Store,ProductCategory,UnitPrice,Quantity,DiscountPercent,GrossSales,NetSales,Cost,Profit,PaymentMethod,CustomerSegment,CustomerAge,CustomerGender,SatisfactionScore,DeliveryDays
3,ORD103293,2023-07-31,West,Store_06,Clothing,184.86,4,0.0,887.33,887.33,400.22,487.11,Debit Card,Returning,45.0,Female,5.0,0.0
5,ORD102906,2024-04-01,Central,Store_06,Clothing,67.37,2,15.0,134.74,114.53,68.90,45.63,Mobile Wallet,New,28.0,Male,3.5,1.0
15,ORD101178,2023-07-09,North,Store_14,Clothing,188.74,2,0.0,452.98,452.98,323.85,129.13,Mobile Wallet,Returning,18.0,Male,5.0,3.0


## 6. Sorting Data

In [20]:
# .sort_values() - sort rows by the values in one or more columns
# ascending=False sorts largest to smallest (descending order)
top_sales = df.sort_values("NetSales", ascending=False)
top_sales[["OrderID", "NetSales"]].head(5)

,OrderID,NetSales
120,ORD104356,14448.077792
1959,ORD100662,10735.127629
2709,ORD100754,9752.910550
4644,ORD100274,6546.180000
1445,ORD100799,5694.460000


In [21]:
# Sort by MULTIPLE columns: first by Region (A-Z), then within each Region by
# NetSales (largest first) - pass a list to "by" and a matching list to "ascending"
multi_sort = df.sort_values(by=["Region", "NetSales"], ascending=[True, False])
multi_sort[["Region", "NetSales"]].head(8)

,Region,NetSales
2926,Central,5000.90
4851,Central,4139.04
5663,Central,3915.87
2600,Central,3651.90
129,Central,3522.37
2698,Central,3390.19
2824,Central,3314.70
4592,Central,3068.02


In [22]:
# .sort_index() - sort rows by their index label rather than by column values
# (useful after operations that leave the index shuffled, e.g. after .sample())
shuffled = df.sample(frac=1, random_state=1)   # shuffle all rows randomly
resorted = shuffled.sort_index()               # put them back in original row order
print("Index is back in order:", list(resorted.index[:5]))

Index is back in order: [0, 1, 2, 3, 4]


## 7. Handling Missing Data

Missing values in pandas are represented as `NaN` (Not a Number) for numeric columns, or
`NaT` (Not a Time) for datetime columns, or `None` for generic objects.

In [23]:
# .isnull() / .isna() (identical, isna is just the newer preferred name) - detect missing values
# .sum() on top counts how many True (missing) values exist per column
df.isna().sum()[df.isna().sum() > 0]

Store                 60
DiscountPercent      122
CustomerAge          241
CustomerGender        90
SatisfactionScore    361
DeliveryDays         182
dtype: int64

In [24]:
# .dropna() - remove rows containing any missing value
# subset=[...] restricts the check to specific columns only
rows_before = len(df)
dropped = df.dropna(subset=["SatisfactionScore"])
print(f"Rows before: {rows_before:,} | after dropping missing SatisfactionScore: {len(dropped):,}")

Rows before: 6,045 | after dropping missing SatisfactionScore: 5,684


In [25]:
# .fillna() - replace missing values instead of dropping the whole row
# Filling with a single fixed value:
filled_fixed = df["DiscountPercent"].fillna(0)

# Filling with a computed statistic (median is often better than mean for skewed data):
filled_median = df["CustomerAge"].fillna(df["CustomerAge"].median())

print("Missing DiscountPercent before:", df["DiscountPercent"].isna().sum())
print("Missing DiscountPercent after filling with 0:", filled_fixed.isna().sum())

Missing DiscountPercent before:

 122
Missing DiscountPercent after filling with 0: 0


In [26]:
# Forward-fill / backward-fill - propagate the last valid value forward or the next
# valid value backward. Common for time-ordered data where a missing reading likely
# resembles its neighbor (e.g. sensor data, stock prices).
example_series = pd.Series([10, np.nan, np.nan, 25, np.nan, 30])
print("Original:      ", list(example_series))
print("Forward-filled:", list(example_series.ffill()))
print("Backward-filled:", list(example_series.bfill()))

Original:       [10.0, nan, nan, 25.0, nan, 30.0]
Forward-filled: [10.0, 10.0, 10.0, 25.0, 25.0, 30.0]
Backward-filled: [10.0, 25.0, 25.0, 25.0, 30.0, 30.0]


## 8. Modifying Data: Adding, Dropping, and Renaming Columns

In [27]:
# Adding a new column is as simple as assigning to a new column name -
# here we compute a new column from two existing ones (vectorized, no loop needed)
df["ProfitMargin"] = (df["Profit"] / df["NetSales"] * 100).round(2)
df[["Profit", "NetSales", "ProfitMargin"]].head(3)

,Profit,NetSales,ProfitMargin
0,1257.31554,2934.99121,42.84
1,14.94000,47.48000,31.47
2,19.08000,42.72000,44.66


In [28]:
# .assign() - add one or more new columns without modifying the original DataFrame in place,
# returning a new DataFrame instead. Useful for chaining multiple operations together fluently.
df_with_extra = df.assign(
    RevenuePerUnit=lambda d: d["NetSales"] / d["Quantity"],
    IsBigOrder=lambda d: d["Quantity"] >= 3
)
df_with_extra[["NetSales", "Quantity", "RevenuePerUnit", "IsBigOrder"]].head(3)

,NetSales,Quantity,RevenuePerUnit,IsBigOrder
0,2934.99121,1,2934.99121,False
1,47.48000,1,47.48000,False
2,42.72000,2,21.36000,False


In [29]:
# .drop() - remove columns (axis=1) or rows (axis=0)
# axis=1 means "operate on columns"; columns=[...] is the modern, clearer alternative
df_fewer_cols = df.drop(columns=["ProfitMargin"])
print("Columns after dropping ProfitMargin:", "ProfitMargin" in df_fewer_cols.columns)

# Dropping specific rows by their index label
df_fewer_rows = df.drop(index=[0, 1, 2])
print(f"Rows after dropping 3 rows: {len(df_fewer_rows):,} (was {len(df):,})")

Columns after dropping ProfitMargin: False


Rows after dropping 3 rows: 6,042 (was 6,045)


In [30]:
# .rename() - rename one or more columns using a dictionary of {old_name: new_name}
renamed = df.rename(columns={"NetSales": "Revenue", "CustomerAge": "Age"})
print(list(renamed.columns[:12]))

['OrderID', 'OrderDate', 'Region', 'Store', 'ProductCategory', 'UnitPrice', 'Quantity', 'DiscountPercent', 'GrossSales', 'Revenue', 'Cost', 'Profit']


## 9. Apply, Map, and Applymap

These three methods run a custom function across data, but at different scopes:

- **`.map()`** - element-wise, on a **Series** only
- **`.apply()`** - can run on a Series (element-wise) OR a DataFrame (row-wise/column-wise)
- **`.applymap()` / `.map()` on a DataFrame** - element-wise across an entire DataFrame

In [31]:
# .map() on a Series - transform each value using a function or a mapping dictionary
segment_labels = {"New": "N", "Returning": "R", "VIP": "V"}
df["SegmentCode"] = df["CustomerSegment"].map(segment_labels)
df[["CustomerSegment", "SegmentCode"]].head(3)

,CustomerSegment,SegmentCode
0,New,N
1,Returning,R
2,New,N


In [32]:
# .apply() on a Series - run a custom function on every value
def classify_price(price):
    """Bucket a unit price into a simple tier label."""
    if price < 20:
        return "Budget"
    elif price < 100:
        return "Mid-range"
    else:
        return "Premium"

df["PriceTier"] = df["UnitPrice"].apply(classify_price)
df[["UnitPrice", "PriceTier"]].head(5)

,UnitPrice,PriceTier
0,374.09,Premium
1,47.48,Mid-range
2,21.36,Mid-range
3,184.86,Premium
4,51.84,Mid-range


In [33]:
# .apply() on a DataFrame with axis=1 - run a function ROW BY ROW, giving access to
# every column of that row at once (useful when a calculation needs multiple columns together)
def order_summary(row):
    """Build a one-line human-readable summary string from a full row of data."""
    return f"{row['Quantity']}x {row['ProductCategory']} for ${row['NetSales']:.2f}"

df["OrderSummary"] = df.apply(order_summary, axis=1)
df["OrderSummary"].head(5)

0    1x Electronics for $2934.99
1           1x Beauty for $47.48
2        2x Groceries for $42.72
3        4x Clothing for $887.33
4    1x Home & Living for $62.21
Name: OrderSummary, dtype: str

## 10. Grouping and Aggregation (`groupby`)

`groupby` follows the classic **split - apply - combine** pattern: split the data into
groups based on a column's values, apply an aggregation function to each group
independently, then combine the results back into a single output.

In [34]:
# Simplest form: group by one column, aggregate one column with one function
avg_sales_by_region = df.groupby("Region", observed=True)["NetSales"].mean().round(2)
avg_sales_by_region.sort_values(ascending=False)

Region
west       840.82
South      420.19
East       384.57
central    365.10
North      342.74
West       342.53
north      340.05
Central    327.03
south      220.36
Name: NetSales, dtype: float64

In [35]:
# Multiple aggregation functions at once using .agg()
region_stats = df.groupby("Region", observed=True)["NetSales"].agg(["count", "sum", "mean", "max"])
region_stats.round(2)

,count,sum,mean,max
Region,,,,
Central,955,312313.55,327.03,5000.90
East,1068,410715.82,384.57,14448.08
North,1329,455501.60,342.74,5694.46
South,1483,623138.36,420.19,10735.13
West,1200,411035.10,342.53,9752.91
central,3,1095.31,365.10,625.98
north,4,1360.20,340.05,906.54
south,1,220.36,220.36,220.36
west,2,1681.65,840.82,1502.88


In [36]:
# Grouping by MULTIPLE columns at once - produces a hierarchical (MultiIndex) result
category_region_profit = df.groupby(["Region", "ProductCategory"], observed=True)["Profit"].sum()
category_region_profit.head(10)

Region   ProductCategory
Central  Beauty              4297.560000
         Clothing           11715.760000
         Electronics        66780.160000
         Groceries           6413.682817
         HOME & LIVING        383.970000
         Home & Living      24188.930959
         Sports             10006.561182
East     BEAUTY                32.520000
         Beauty              6577.340000
         Clothing           14595.869119
Name: Profit, dtype: float64

In [37]:
# Named aggregation: apply DIFFERENT functions to DIFFERENT columns in one call,
# with clean, custom output column names (the modern, most readable way to do this)
summary = df.groupby("CustomerSegment", observed=True).agg(
    total_orders=("OrderID", "count"),
    total_revenue=("NetSales", "sum"),
    avg_satisfaction=("SatisfactionScore", "mean"),
    avg_delivery_days=("DeliveryDays", "mean")
).round(2)
summary

,total_orders,total_revenue,avg_satisfaction,avg_delivery_days
CustomerSegment,,,,
New,2132,801874.42,3.96,3.58
Returning,2715,959132.82,3.95,3.46
VIP,1198,456054.71,3.92,3.65


In [38]:
# .transform() - unlike .agg() (which collapses each group to one row), .transform()
# returns a result the SAME LENGTH as the original DataFrame, broadcasting the group's
# aggregate value back onto every row that belongs to that group. Very useful for
# creating features like "this order's sales vs. its category's average sales".
df["CategoryAvgSales"] = df.groupby("ProductCategory", observed=True)["NetSales"].transform("mean")
df["SalesVsCategoryAvg"] = df["NetSales"] - df["CategoryAvgSales"]
df[["ProductCategory", "NetSales", "CategoryAvgSales", "SalesVsCategoryAvg"]].head(5)

,ProductCategory,NetSales,CategoryAvgSales,SalesVsCategoryAvg
0,Electronics,2934.99121,1054.523153,1880.468057
1,Beauty,47.48000,127.908071,-80.428071
2,Groceries,42.72000,55.331706,-12.611706
3,Clothing,887.33000,183.130814,704.199186
4,Home & Living,62.21000,481.684731,-419.474731


## 11. Pivot Tables and Cross-Tabulation

In [39]:
# .pivot_table() - reshape data so one column's unique values become new columns,
# with an aggregation function combining any resulting duplicates. This is the pandas
# equivalent of an Excel PivotTable.
pivot = df.pivot_table(
    values="NetSales",
    index="Region",             # rows
    columns="ProductCategory",  # columns
    aggfunc="sum",
    fill_value=0,                # replace any empty combinations with 0 instead of NaN
    observed=True
)
pivot.round(0)

ProductCategory,BEAUTY,Beauty,Clothing,ELECTRONICS,Electronics,GROCERIES,Groceries,HOME & LIVING,Home & Living,SPORTS,Sports
Region,,,,,,,,,,,
Central,0.0,10722.0,29073.0,0.0,168038.0,0.0,16511.0,876.0,60797.0,0.0,26297.0
East,61.0,16238.0,36058.0,0.0,215569.0,34.0,16312.0,1001.0,97005.0,0.0,28438.0
North,0.0,16678.0,43757.0,0.0,268465.0,0.0,23150.0,545.0,77090.0,84.0,25731.0
South,0.0,24243.0,40856.0,0.0,378467.0,38.0,20284.0,0.0,116907.0,0.0,42343.0
West,0.0,18457.0,36797.0,839.0,244527.0,0.0,19658.0,0.0,64032.0,0.0,26725.0
central,0.0,0.0,184.0,0.0,0.0,0.0,0.0,0.0,286.0,0.0,626.0
north,0.0,0.0,436.0,0.0,907.0,0.0,18.0,0.0,0.0,0.0,0.0
south,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,220.0
west,0.0,0.0,0.0,0.0,0.0,0.0,179.0,0.0,1503.0,0.0,0.0


In [40]:
# pivot_table with multiple aggregation functions and a margin (grand total) row/column
pivot_with_totals = df.pivot_table(
    values="NetSales",
    index="Region",
    columns="CustomerSegment",
    aggfunc="sum",
    margins=True,        # adds an "All" row and column with grand totals
    margins_name="Total",
    observed=True
)
pivot_with_totals.round(0)

CustomerSegment,New,Returning,VIP,Total
Region,,,,
Central,113497.0,141370.0,57446.0,312314.0
East,156712.0,154835.0,99169.0,410716.0
North,165369.0,212681.0,77451.0,455502.0
South,212146.0,266022.0,144970.0,623138.0
West,153751.0,180552.0,76733.0,411035.0
central,NaN,810.0,286.0,1095.0
north,NaN,1360.0,NaN,1360.0
south,220.0,NaN,NaN,220.0
west,179.0,1503.0,NaN,1682.0


In [41]:
# pd.crosstab() - a specialized shortcut for frequency counts between two categorical
# columns (like pivot_table but defaults to counting rows rather than needing a "values" column)
crosstab_result = pd.crosstab(df["Region"], df["PaymentMethod"])
crosstab_result

PaymentMethod,Bank Transfer,Cash,Credit Card,Debit Card,Mobile Wallet
Region,,,,,
Central,180,204,191,185,195
East,224,228,182,224,210
North,252,257,281,256,283
South,311,299,291,281,301
West,246,249,241,227,237
central,0,1,1,1,0
north,1,0,2,0,1
south,0,0,1,0,0
west,0,0,0,1,1


## 12. Merging, Joining, and Concatenating

Combining multiple DataFrames is one of the most common real-world pandas tasks - just like
SQL JOINs, or stacking spreadsheets on top of each other.

In [42]:
# Build two small example DataFrames to demonstrate merging clearly
store_info = pd.DataFrame({
    "Store": ["Store_01", "Store_02", "Store_03"],
    "Manager": ["Amaka Obi", "David Chen", "Fatima Bello"],
    "OpenedYear": [2015, 2018, 2020]
})

# pd.merge() - combine two DataFrames based on a shared key column, like a SQL JOIN
# how="left" keeps every row from the LEFT DataFrame (df), filling unmatched rows with NaN
merged = pd.merge(df, store_info, on="Store", how="left")
merged[["OrderID", "Store", "Manager", "OpenedYear"]].head(6)

,OrderID,Store,Manager,OpenedYear
0,ORD103076,Store_06,NaN,NaN
1,ORD104130,Store_19,NaN,NaN
2,ORD104684,Store_03,Fatima Bello,2020.0
3,ORD103293,Store_06,NaN,NaN
4,ORD102397,Store_19,NaN,NaN
5,ORD102906,Store_06,NaN,NaN


**Join types in `pd.merge()`, explained:**
- `how="inner"` - keep only rows where the key exists in BOTH DataFrames
- `how="left"` - keep all rows from the left DataFrame, fill unmatched right-side columns with NaN
- `how="right"` - keep all rows from the right DataFrame, fill unmatched left-side columns with NaN
- `how="outer"` - keep every row from BOTH DataFrames, filling gaps with NaN wherever no match exists

In [43]:
# pd.concat() - stack DataFrames together, either vertically (more rows) or
# horizontally (more columns). axis=0 (default) stacks rows; axis=1 stacks columns side by side.
q1_orders = df[df["OrderDate"].dt.quarter == 1].head(3)
q2_orders = df[df["OrderDate"].dt.quarter == 2].head(3)

# Stack vertically: combine two separate row-batches into one DataFrame
stacked = pd.concat([q1_orders, q2_orders], axis=0)
print(f"Q1 sample: {len(q1_orders)} rows | Q2 sample: {len(q2_orders)} rows | Combined: {len(stacked)} rows")

Q1 sample: 3 rows | Q2 sample: 3 rows | Combined: 6 rows


In [44]:
# .join() - a convenience method for merging on the INDEX rather than a column
# (functionally similar to merge, but index-based by default)
left_df = pd.DataFrame({"Value": [1, 2, 3]}, index=["a", "b", "c"])
right_df = pd.DataFrame({"Label": ["X", "Y", "Z"]}, index=["a", "b", "c"])

joined = left_df.join(right_df)
joined

,Value,Label
a,1,X
b,2,Y
c,3,Z


## 13. Reshaping Data: Melt, Pivot, Stack/Unstack

In [45]:
# .melt() - convert data from WIDE format to LONG format
# (the opposite of pivot_table). Very useful before plotting with seaborn/plotly,
# which usually expect "long" (tidy) data with one row per observation.
wide_data = pivot.reset_index()   # our earlier Region x ProductCategory pivot table
print("WIDE format:")
print(wide_data.head(2))

long_data = wide_data.melt(id_vars="Region", var_name="ProductCategory", value_name="TotalSales")
print("\nLONG format (tidy):")
long_data.head(6)

WIDE format:
ProductCategory   Region  BEAUTY    Beauty      Clothing  ELECTRONICS  Electronics  GROCERIES     Groceries  HOME & LIVING  Home & Living  SPORTS  \
0                Central    0.00  10721.76  29072.570000          0.0    168037.92       0.00  16510.614406         875.96   60797.470240     0.0   
1                   East   60.57  16238.23  36057.964731          0.0    215569.27      34.36  16311.561248        1000.55   97004.880603     0.0   

ProductCategory        Sports  
0                26297.254152  
1                28438.433699  

LONG format (tidy):


,Region,ProductCategory,TotalSales
0,Central,BEAUTY,0.00
1,East,BEAUTY,60.57
2,North,BEAUTY,0.00
3,South,BEAUTY,0.00
4,West,BEAUTY,0.00
5,central,BEAUTY,0.00


In [46]:
# .pivot() - the inverse of melt: turn long data back into wide format.
# (Unlike pivot_table, plain .pivot() does NOT aggregate - it assumes each
# index/column combination appears only once, or it will raise an error.)
back_to_wide = long_data.pivot(index="Region", columns="ProductCategory", values="TotalSales")
back_to_wide.round(0)

ProductCategory,BEAUTY,Beauty,Clothing,ELECTRONICS,Electronics,GROCERIES,Groceries,HOME & LIVING,Home & Living,SPORTS,Sports
Region,,,,,,,,,,,
Central,0.0,10722.0,29073.0,0.0,168038.0,0.0,16511.0,876.0,60797.0,0.0,26297.0
East,61.0,16238.0,36058.0,0.0,215569.0,34.0,16312.0,1001.0,97005.0,0.0,28438.0
North,0.0,16678.0,43757.0,0.0,268465.0,0.0,23150.0,545.0,77090.0,84.0,25731.0
South,0.0,24243.0,40856.0,0.0,378467.0,38.0,20284.0,0.0,116907.0,0.0,42343.0
West,0.0,18457.0,36797.0,839.0,244527.0,0.0,19658.0,0.0,64032.0,0.0,26725.0
central,0.0,0.0,184.0,0.0,0.0,0.0,0.0,0.0,286.0,0.0,626.0
north,0.0,0.0,436.0,0.0,907.0,0.0,18.0,0.0,0.0,0.0,0.0
south,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,220.0
west,0.0,0.0,0.0,0.0,0.0,0.0,179.0,0.0,1503.0,0.0,0.0


In [47]:
# .stack() / .unstack() - pivot between a wider DataFrame and a MultiIndex Series/DataFrame
# stack(): moves the innermost COLUMN level into the innermost ROW index level
# unstack(): the reverse - moves a ROW index level back out into columns
grouped_multi = df.groupby(["Region", "CustomerSegment"], observed=True)["NetSales"].sum()
print("Original (MultiIndex Series):")
print(grouped_multi.head(6))

unstacked = grouped_multi.unstack()   # CustomerSegment moves from row index to columns
print("\nAfter .unstack() (wide format):")
unstacked.round(0)

Original (MultiIndex Series):
Region   CustomerSegment
Central  New                113496.804406
         Returning          141370.284392
         VIP                 57446.460000
East     New                156712.250603
         Returning          154834.757846
         VIP                 99168.811832
Name: NetSales, dtype: float64

After .unstack() (wide format):


CustomerSegment,New,Returning,VIP
Region,,,
Central,113497.0,141370.0,57446.0
East,156712.0,154835.0,99169.0
North,165369.0,212681.0,77451.0
South,212146.0,266022.0,144970.0
West,153751.0,180552.0,76733.0
central,NaN,810.0,286.0
north,NaN,1360.0,NaN
south,220.0,NaN,NaN
west,179.0,1503.0,NaN


## 14. String Operations

The `.str` accessor gives vectorized access to Python string methods across an entire
column at once, without needing a manual loop.

In [48]:
# .str.upper() / .str.lower() / .str.title() - change text casing
print(df["Region"].str.upper().unique()[:5])
print(df["Region"].str.lower().unique()[:5])

<StringArray>
['SOUTH', 'WEST', 'CENTRAL', 'NORTH', 'EAST']
Length: 5, dtype: str
<StringArray>
['south', 'west', 'central', 'north', 'east']
Length: 5, dtype: str


In [49]:
# .str.contains() - boolean mask for substring matching (supports regex too)
mobile_payments = df[df["PaymentMethod"].str.contains("Mobile", case=False)]
print(f"Mobile Wallet orders: {len(mobile_payments):,}")

# .str.startswith() / .str.endswith()
store_1x = df[df["Store"].str.startswith("Store_1")]
print(f"Orders from Store_10 through Store_19: {len(store_1x):,}")

Mobile Wallet orders: 1,228
Orders from Store_10 through Store_19: 3,014


In [50]:
# .str.len() - length of each string; .str.replace() - substring replacement;
# .str.split() - split each string into a list on a delimiter
order_id_lengths = df["OrderID"].str.len()
print("All OrderIDs same length:", order_id_lengths.nunique() == 1)

cleaned_ids = df["OrderID"].str.replace("ORD", "Order-")
print(cleaned_ids.head(3).tolist())

split_example = pd.Series(["North-Store01", "South-Store02"]).str.split("-", expand=True)
split_example.columns = ["Region", "StoreCode"]
split_example

All OrderIDs same length: True
['Order-103076', 'Order-104130', 'Order-104684']


,Region,StoreCode
0,North,Store01
1,South,Store02


## 15. Date and Time Handling

Pandas has extensive, purpose-built support for datetime data through the `.dt` accessor
and specialized time-series functions.

In [51]:
# .dt accessor - extract individual date components from a datetime column
df["Year"] = df["OrderDate"].dt.year
df["Month"] = df["OrderDate"].dt.month
df["MonthName"] = df["OrderDate"].dt.month_name()
df["DayOfWeek"] = df["OrderDate"].dt.day_name()
df["Quarter"] = df["OrderDate"].dt.quarter
df["IsWeekend"] = df["OrderDate"].dt.dayofweek >= 5   # Saturday=5, Sunday=6

df[["OrderDate", "Year", "Month", "MonthName", "DayOfWeek", "Quarter", "IsWeekend"]].head(5)

,OrderDate,Year,Month,MonthName,DayOfWeek,Quarter,IsWeekend
0,2023-03-21,2023,3,March,Tuesday,1,False
1,2023-01-07,2023,1,January,Saturday,1,True
2,2024-01-13,2024,1,January,Saturday,1,True
3,2023-07-31,2023,7,July,Monday,3,False
4,2023-07-31,2023,7,July,Monday,3,False


In [52]:
# Date arithmetic - subtracting two dates gives a Timedelta;
# adding a pd.Timedelta or pd.DateOffset shifts a date forward/backward
df["DaysSinceOrder"] = (pd.Timestamp("2025-01-01") - df["OrderDate"]).dt.days
df["EstimatedDeliveryDate"] = df["OrderDate"] + pd.to_timedelta(df["DeliveryDays"], unit="D")

df[["OrderDate", "DeliveryDays", "EstimatedDeliveryDate", "DaysSinceOrder"]].head(5)

,OrderDate,DeliveryDays,EstimatedDeliveryDate,DaysSinceOrder
0,2023-03-21,1.0,2023-03-22,652
1,2023-01-07,3.0,2023-01-10,725
2,2024-01-13,1.0,2024-01-14,354
3,2023-07-31,0.0,2023-07-31,520
4,2023-07-31,4.0,2023-08-04,520


In [53]:
# .resample() - group time-series data into fixed time buckets (like groupby, but for dates)
# "ME" = Month End, "W" = Weekly, "Q" = Quarterly, "D" = Daily, etc.
monthly_totals = df.set_index("OrderDate").resample("ME")["NetSales"].sum()
monthly_totals.head(6)

OrderDate
2023-01-31     84234.417629
2023-02-28     71893.130000
2023-03-31     94313.917107
2023-04-30     78900.314523
2023-05-31    100902.060000
2023-06-30     92312.170000
Freq: ME, Name: NetSales, dtype: float64

In [54]:
# Filtering by date range with plain comparison operators works directly on datetime columns
date_range_orders = df[(df["OrderDate"] >= "2023-11-01") & (df["OrderDate"] <= "2023-12-31")]
print(f"Orders in Nov-Dec 2023: {len(date_range_orders):,}")

# .between() is a cleaner alternative for the same kind of range filter
date_range_orders_v2 = df[df["OrderDate"].between("2023-11-01", "2023-12-31")]
print(f"Same result using .between(): {len(date_range_orders_v2):,}")

Orders in Nov-Dec 2023: 517
Same result using .between(): 517


## 16. Categorical Data

The `category` dtype is a memory- and speed-optimized way to store columns with a small,
repeated set of possible values (as opposed to `object`/string dtype, which stores every
value independently even when many rows repeat the same text).

In [55]:
# Converting a text column to category dtype, and comparing memory usage before/after
memory_as_object = df["ProductCategory"].astype("object").memory_usage(deep=True)
memory_as_category = df["ProductCategory"].astype("category").memory_usage(deep=True)

print(f"Memory as object dtype:   {memory_as_object:,} bytes")
print(f"Memory as category dtype: {memory_as_category:,} bytes")
print(f"Reduction: {(1 - memory_as_category/memory_as_object)*100:.1f}%")

Memory as object dtype:   352,023 bytes
Memory as category dtype: 6,814 bytes
Reduction: 98.1%


In [56]:
# Ordered categories - useful when categories have a natural rank/order
# (e.g. Low < Medium < High), which then makes sorting and comparison operators
# ( <, >, etc.) behave meaningfully instead of alphabetically
tier_order = pd.CategoricalDtype(categories=["Low", "Medium", "High"], ordered=True)
sample_tiers = pd.Series(["Medium", "Low", "High", "Medium"]).astype(tier_order)

print(sample_tiers)
print("\nSorted respecting the defined order (not alphabetical):")
print(sample_tiers.sort_values())
print("\nIs 'High' greater than 'Low'?", sample_tiers.iloc[2] > sample_tiers.iloc[1])

0    Medium
1       Low
2      High
3    Medium
dtype: category
Categories (3, str): ['Low' < 'Medium' < 'High']

Sorted respecting the defined order (not alphabetical):
1       Low
0    Medium
3    Medium
2      High
dtype: category
Categories (3, str): ['Low' < 'Medium' < 'High']

Is 'High' greater than 'Low'? False


## 17. Window Functions: Rolling, Expanding, and Cumulative

In [57]:
# .rolling(window=N) - compute a statistic over a sliding window of N consecutive rows
# Commonly used to smooth out noisy time series (e.g. a 7-day moving average)
daily_sales = df.set_index("OrderDate").resample("D")["NetSales"].sum()

rolling_7day = daily_sales.rolling(window=7).mean()
print("Daily sales (first 10 days):")
print(daily_sales.head(10))
print("\n7-day rolling average (first 10 days - first 6 are NaN, not enough data yet):")
print(rolling_7day.head(10))

Daily sales (first 10 days):
OrderDate
2023-01-01    2807.85
2023-01-02    1583.09
2023-01-03    2681.16
2023-01-04    3487.34
2023-01-05    1928.34
2023-01-06    1909.64
2023-01-07    1062.16
2023-01-08    1391.70
2023-01-09    2444.08
2023-01-10     677.82
Freq: D, Name: NetSales, dtype: float64

7-day rolling average (first 10 days - first 6 are NaN, not enough data yet):
OrderDate
2023-01-01            NaN
2023-01-02            NaN
2023-01-03            NaN
2023-01-04            NaN
2023-01-05            NaN
2023-01-06            NaN
2023-01-07    2208.511429
2023-01-08    2006.204286
2023-01-09    2129.202857
2023-01-10    1843.011429
Freq: D, Name: NetSales, dtype: float64


In [58]:
# .expanding() - like .rolling(), but the window grows to include ALL prior rows each time
# (a running/cumulative statistic up to each point), rather than a fixed-size window
expanding_avg = daily_sales.expanding().mean()
print("Expanding (running) average of daily sales:")
expanding_avg.head(6)

Expanding (running) average of daily sales:


OrderDate
2023-01-01    2807.850000
2023-01-02    2195.470000
2023-01-03    2357.366667
2023-01-04    2639.860000
2023-01-05    2497.556000
2023-01-06    2399.570000
Freq: D, Name: NetSales, dtype: float64

In [59]:
# Cumulative functions - .cumsum(), .cummax(), .cummin(), .cumprod()
example = pd.Series([100, 250, 80, 400, 150])
print("Original:      ", list(example))
print("Cumulative sum:", list(example.cumsum()))
print("Cumulative max:", list(example.cummax()))

Original:       [100, 250, 80, 400, 150]
Cumulative sum: [100, 350, 430, 830, 980]
Cumulative max: [100, 250, 250, 400, 400]


## 18. MultiIndex (Hierarchical Indexing)

A MultiIndex lets a DataFrame or Series be indexed by more than one level at once - this
happens automatically whenever you `groupby` more than one column, but it can also be
built and used deliberately.

In [60]:
# groupby on 2 columns automatically produces a MultiIndex, as seen earlier
multi_indexed = df.groupby(["Region", "ProductCategory"], observed=True)["NetSales"].sum()
print("Index type:", type(multi_indexed.index))
print("\nIndex levels:", multi_indexed.index.names)
multi_indexed.head(6)

Index type: <class 'pandas.MultiIndex'>

Index levels: ['Region', 'ProductCategory']


Region   ProductCategory
Central  Beauty              10721.760000
         Clothing            29072.570000
         Electronics        168037.920000
         Groceries           16510.614406
         HOME & LIVING         875.960000
         Home & Living       60797.470240
Name: NetSales, dtype: float64

In [61]:
# Selecting from a MultiIndex: .loc[] with a tuple selects an exact combination,
# while .loc["level_1_value"] alone selects every row under that top level
print("All 'North' region rows:")
print(multi_indexed.loc["North"])

print("\nExact combination (North, Electronics):")
print(multi_indexed.loc[("North", "Electronics")])

All 'North' region rows:
ProductCategory
Beauty            16678.260000
Clothing          43757.150000
Electronics      268465.230000
Groceries         23149.737987
HOME & LIVING       545.260000
Home & Living     77090.260000
SPORTS               84.250000
Sports            25731.450000
Name: NetSales, dtype: float64

Exact combination (North, Electronics):
268465.23


In [62]:
# .reset_index() - flatten a MultiIndex (or any index) back into plain columns,
# which is often the most convenient format for further filtering, merging, or plotting
flat = multi_indexed.reset_index()
flat.columns = ["Region", "ProductCategory", "TotalNetSales"]
flat.head(6)

,Region,ProductCategory,TotalNetSales
0,Central,Beauty,10721.760000
1,Central,Clothing,29072.570000
2,Central,Electronics,168037.920000
3,Central,Groceries,16510.614406
4,Central,HOME & LIVING,875.960000
5,Central,Home & Living,60797.470240


## 19. Exporting Data

Just as pandas can read many formats, it can write to just as many, using the mirror-image
`.to_...()` methods.

In [63]:
# .to_csv() - write to a CSV file. index=False avoids writing the DataFrame's row index
# as an extra unwanted column in the output file (a very common beginner mistake to forget)
summary.to_csv("pandas_guide_groupby_summary.csv", index=True)
print("Saved: pandas_guide_groupby_summary.csv")

# .to_excel() - write to an Excel file, optionally specifying the sheet name
pivot.to_excel("pandas_guide_pivot_output.xlsx", sheet_name="RegionCategorySales")
print("Saved: pandas_guide_pivot_output.xlsx")

# .to_json() / .to_dict() / .to_parquet() follow the same pattern for other formats
summary_dict = summary.reset_index().to_dict(orient="records")
print("\nFirst record as a plain Python dict:")
print(summary_dict[0])

Saved: pandas_guide_groupby_summary.csv


Saved: pandas_guide_pivot_output.xlsx

First record as a plain Python dict:
{'CustomerSegment': 'New', 'total_orders': 2132, 'total_revenue': 801874.42, 'avg_satisfaction': 3.96, 'avg_delivery_days': 3.58}


## 20. Performance Tips and Best Practices

**1. Avoid Python `for` loops over rows - use vectorized operations instead.**
Vectorized operations (working on entire columns at once) run in optimized C code under
the hood and are dramatically faster than looping row by row in Python.

In [64]:
import time

# SLOW approach: looping row by row with .iterrows()
start = time.time()
results_slow = []
for idx, row in df.head(2000).iterrows():
    results_slow.append(row["UnitPrice"] * row["Quantity"])
slow_time = time.time() - start

# FAST approach: vectorized column-wise multiplication (no explicit loop at all)
start = time.time()
results_fast = df.head(2000)["UnitPrice"] * df.head(2000)["Quantity"]
fast_time = time.time() - start

print(f"Loop-based approach:      {slow_time*1000:.2f} ms")
print(f"Vectorized approach:      {fast_time*1000:.2f} ms")
print(f"Vectorized was ~{slow_time/fast_time:.0f}x faster")

Loop-based approach:      79.32 ms
Vectorized approach:      2.52 ms
Vectorized was ~31x faster


**Other key performance and best-practice guidelines:**

2. **Use `.loc`/`.iloc` instead of chained indexing** (e.g. `df[df.x > 0]["y"] = 1` can
   trigger pandas' `SettingWithCopyWarning` and may silently fail to update the original
   data - use `df.loc[df.x > 0, "y"] = 1` instead).

3. **Convert low-cardinality text columns to `category` dtype** (Section 16) to reduce
   memory usage and speed up groupby/merge operations significantly on large datasets.

4. **Read only the columns you need** with `pd.read_csv(..., usecols=[...])` when working
   with very wide files, rather than loading everything and dropping columns afterward.

5. **Use `.query()` or vectorized boolean masks** rather than `.apply()` with a custom
   Python function whenever a plain comparison/arithmetic expression can do the same job -
   `.apply()` still loops in Python under the hood and is much slower than true vectorization.

6. **Chain methods thoughtfully** (e.g. `df.dropna().sort_values().reset_index()`) for
   cleaner, more readable code - but break very long chains into intermediate named
   variables when it helps readability or debugging.

7. **Downcast numeric types** (e.g. `float64` to `float32`, or `int64` to `int32`/`int16`
   when the value range allows) via `pd.to_numeric(col, downcast="float")` to cut memory
   usage further on very large datasets.

---
## Summary

This notebook covered the essential pandas toolkit end to end: creating and inspecting
Series/DataFrames, selecting and filtering data, cleaning missing values, transforming
columns with apply/map, aggregating with groupby and pivot tables, combining datasets with
merge/concat/join, reshaping with melt/pivot/stack, working with strings and dates,
categorical data, window functions, MultiIndex, exporting, and performance best practices.

Together with the earlier EDA notebook, this gives a complete practical foundation for
using pandas in real data science and analytics work.

**Kajola Gbenga**
**Data Scientist & AI Engineer**
